# hrsmpipe walkthrough

Load a run config, run the pipeline, then look at the metrics and the paper-style figure for one site.

Everything the package does is defined by one TOML file; `config.resolved.toml` in the run directory records the fully expanded config and the git provenance. Unchanged stages are skipped, so re-running this notebook is cheap.

Needs `HRSM_DATA_DIR` (or `run.data_root`), and Earth Engine access for a first run (`gee.project`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from hrsmpipe.config import load_config
from hrsmpipe.evaluate import plots
from hrsmpipe.paths import RunPaths, resolve_data_root
from hrsmpipe.pipeline import run_stages
from hrsmpipe.sites import load_sites


In [ ]:
CONFIG = Path("configs/study_sites.toml")  # or configs/wisconet.toml
# dotted-key overrides of single fields, e.g. {"run.name": "demo", "sites.include": [296]}
OVERRIDES = {}
SITE_ID = 296  # which site to plot

cfg = load_config(CONFIG, OVERRIDES)
sites = load_sites(cfg)
paths = RunPaths(resolve_data_root(cfg), cfg.run.name)
print(f"run {cfg.run.name!r}: {cfg.run.start} -> {cfg.run.end} (UTC), {len(sites)} sites, outputs in {paths.root}")

## Run the stages

`run_stages` takes `from_`, `to`, `force` (rerun that stage and everything after it) and `sites`. The stages are `fetch-static`, `fetch-rs`, `fetch-obs`, `features`, `infer`, `postproc`, `forcing`, `lgar`, `evaluate`.

In [ ]:
report = run_stages(cfg)
print("ran:", report.ran, "| skipped:", report.skipped)

## Metrics vs observations

In [ ]:
metrics = pd.read_csv(paths.root / "evaluate" / "metrics.csv")
site_metrics = metrics[(metrics.site_id == SITE_ID) & ~metrics.series.str.endswith("_daily")]
display(site_metrics.set_index(["layer", "series"]).drop(columns=["site_id", "station_code"]).round(4))

## Paper-style figure for one site

ML inference (blue triangles), daily-mean observations (red), the LGAR+EnKF analysis ±1σ and the open-loop LGAR run, back-transformed to the ML climatology, with daily precipitation as inverted bars.

In [ ]:
site = pd.read_parquet(paths.root / "lgar" / f"site_{SITE_ID}.parquet")
fig = plots.plot_site_paper(site, SITE_ID, sites.code(SITE_ID), cfg.evaluate.time_window or None, figsize=(12, 8))
display(fig)
plt.close(fig)

`site` is the merged per-site table (`lgar/site_<id>.parquet`): LGAR and EnKF outputs, the post-processed ML, observations, precipitation and PET with per-hour source flags.

In [ ]:
site.filter(["analysis_sm_sly", "analysis_sm_rly", "observation_sly", "observation_rly", "precip_mm", "precip_source"]).dropna(how="all").head()